# CARLA allein in Google Colab prüfen

Wähle unter **Laufzeit → Laufzeittyp ändern** eine GPU und führe alle Zellen aus.
Dieses Notebook installiert den CARLA-Client, NumPy, Pillow und die benötigten
Linux-/Video-Pakete. Es benötigt keine Modellgewichte, Bridge-Installation oder Anmeldung.

1. **cpu-rpc:** CARLA mit deaktiviertem Renderer starten und RPC/Weltzugriff prüfen.
   Ein Erfolg hier bestätigt noch keine Kamerafunktion.
2. **gpu-rgb:** Vulkan-Rendering starten, ein Fahrzeug mit einer RGB-Kamera spawnen
   und 30 Kameraframes bei 10 Hz erfassen, davon mindestens 15 mit unterschiedlichem
   Bildinhalt. Die kurze Fahrt benutzt
   einen festen Gasbefehl, damit wir nur CARLA und dessen Kameras prüfen.
3. Das Kameravideo anzeigen und Diagnosen als ZIP herunterladen — auch bei Fehlern.

Eine frische Laufzeit benötigt etwa **8,35 GB Download** und mindestens **35 GB freien
Speicher**. In derselben Laufzeit wird die Installation des bisherigen Alpamayo-Notebooks
unter /content/alpamayo-bridge-check wiederverwendet. Der GPU-Test startet mit CARLAs
Standardqualität; so prüfen wir auch, ob der bisherige Start mit Low für den Fehler
eine Rolle spielt. CARLA 0.9.16 nutzt Vulkan.

In [ ]:
MAP = "Town01"  # @param ["Town01", "Town03"]
RENDER_VIDEO = True  # @param {type:"boolean"}
STARTUP_TIMEOUT = 180  # @param {type:"integer"}
assert STARTUP_TIMEOUT > 0

## 1. Kleine Client-Umgebung und Linux-Pakete installieren

Eine eigene Python-3.12-Umgebung hält die Colab-Kernel-Pakete unverändert.
CARLA selbst wird im ersten Test installiert, falls es noch fehlt.

In [ ]:
import json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

ROOT = Path('/content/alpamayo-bridge-check')
REPORT = ROOT / f'carla-only-report-{time.time_ns()}'
REPORT.mkdir(parents=True, exist_ok=True)
RESULTS = {}
SETUP_OK = False

def run_check(name, command, required=True, **kwargs):
    print(name, '...', flush=True)
    started = time.monotonic()
    completed = subprocess.run(command, capture_output=True, text=True, **kwargs)
    (REPORT / f'{name}.log').write_text(completed.stdout + completed.stderr)
    RESULTS[name] = {'status': 'PASSED' if completed.returncode == 0 else 'FAILED',
                     'returncode': completed.returncode,
                     'seconds': round(time.monotonic() - started, 2)}
    print(completed.stdout[-3000:], completed.stderr[-3000:])
    if completed.returncode and required:
        raise RuntimeError(f'{name} failed; see {name}.log')
    return completed

def run_mode(mode):
    output = REPORT / mode
    if output.exists():
        shutil.rmtree(output)  # A repeated cell must not reuse an earlier result/video.
    try:
        completed = run_check(f'{mode}-worker', [CLIENT_PYTHON, str(ROOT / 'carla-only-worker.py'),
                              str(ROOT), str(REPORT), mode], required=False)
        result = output / 'result.json'
        RESULTS[mode] = json.loads(result.read_text()) if result.exists() else {
            'status': 'FAILED', 'error': f'Worker produced no result.json; see {mode}-worker.log',
            'returncode': completed.returncode}
    except OSError as error:
        RESULTS[mode] = {'status': 'FAILED', 'error': str(error)}
    print(RESULTS[mode])

try:
    if platform.system() != 'Linux' or os.geteuid() != 0:
        raise RuntimeError('Dieses Notebook benötigt die Linux-Laufzeit von Google Colab.')
    if shutil.which('uv') is None:
        run_check('install-uv', [sys.executable, '-m', 'pip', 'install', 'uv'])
    uv = shutil.which('uv')
    run_check('python312', [uv, 'python', 'install', '3.12'])
    environment = ROOT / '.venv-carla-only'
    if not environment.exists():
        run_check('client-env', [uv, 'venv', '--python', '3.12', str(environment)])
    CLIENT_PYTHON = str(environment / 'bin/python')
    run_check('client-packages', [uv, 'pip', 'install', '--python', CLIENT_PYTHON,
              'carla==0.9.16', 'numpy>=1.24', 'pillow>=10'])
    run_check('apt-update', ['apt-get', 'update', '-qq'])
    run_check('runtime-libs', ['apt-get', 'install', '-y', '-qq', 'libvulkan1', 'vulkan-tools',
              'libsdl2-2.0-0', 'libomp5', 'xdg-user-dirs', 'libnss3', 'libglu1-mesa',
              'libegl1', 'libopengl0', 'ffmpeg'])
    (ROOT / 'carla-only-settings.json').write_text(json.dumps({
        'map': MAP, 'render_video': RENDER_VIDEO, 'startup_timeout': STARTUP_TIMEOUT}))
    SETUP_OK = True
except Exception as error:
    RESULTS['setup'] = {'status': 'FAILED', 'error': str(error)}
    print('Setup fehlgeschlagen:', error, 'Die letzte Zelle exportiert trotzdem den Bericht.')

## 2. Testprogramm vorbereiten

Der CPU-Test deaktiviert den Renderer ausdrücklich. Der GPU-Test verlangt echte,
nicht einfarbige RGB-Bilder, unterschiedliche Bildinhalte und Fahrzeugbewegung.
Bei einem Timeout werden GPU-Belegung und Prozesszustände vor dem Abschalten
protokolliert. Engine-Logs werden übernommen, wenn der ausgelieferte Build sie erzeugt.

In [ ]:
CARLA_TEST = r'''
import ctypes.util
import hashlib
import json
import os
from pathlib import Path
import pwd
from queue import Queue
import shutil
import signal
import socket
import subprocess
import sys
import time
import traceback

import carla
import numpy as np
from PIL import Image


def diagnostic(command, path, env=None):
    try:
        result = subprocess.run(command, env=env, capture_output=True, text=True, timeout=15)
        path.write_text(result.stdout + result.stderr)
        return result
    except (OSError, subprocess.TimeoutExpired) as error:
        path.write_text(str(error))
        return None


def stop_server(process):
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        process.wait()
        return
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        pass
    try:
        os.killpg(process.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    process.wait()


def wait_for_server(process, client, timeout, info):
    deadline = time.monotonic() + timeout
    last_error = 'No RPC attempt completed'
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError(f'Engine exited with code {process.returncode}; see server.log')
        try:
            info['phase'] = 'rpc'
            version = client.get_server_version()
            if not version.startswith('0.9.16'):
                raise ValueError(f'Unexpected CARLA version: {version}')
            info.update(rpc='PASSED', server_version=version)
            info['phase'] = 'world'
            return client.get_world()
        except RuntimeError as error:
            last_error = str(error)
            time.sleep(1)
    raise TimeoutError(f"CARLA {info['phase']} not ready after {timeout}s: {last_error}")


def test_rgb(world, output, render_video):
    settings = world.get_settings()
    settings.synchronous_mode = True
    settings.fixed_delta_seconds = 0.1
    world.apply_settings(settings)
    library = world.get_blueprint_library()
    vehicle = world.spawn_actor(library.find('vehicle.tesla.model3'), world.get_map().get_spawn_points()[0])
    camera = None
    try:
        blueprint = library.find('sensor.camera.rgb')
        blueprint.set_attribute('image_size_x', '640')
        blueprint.set_attribute('image_size_y', '360')
        camera = world.spawn_actor(blueprint, carla.Transform(carla.Location(x=1.5, z=1.6)), attach_to=vehicle)
        frames = Queue()
        camera.listen(frames.put)
        vehicle.apply_control(carla.VehicleControl(throttle=0.4))
        start = vehicle.get_location()
        frame_ids, hashes = [], set()
        folder = output / 'frames'
        folder.mkdir()
        # GPU camera readback can lag the world tick; warm up before waiting.
        for _ in range(5):
            world.tick()
        for index in range(30):
            world.tick()
            frame = frames.get(timeout=15)
            rgb = np.frombuffer(frame.raw_data, dtype=np.uint8).reshape(frame.height, frame.width, 4)[:, :, :3][:, :, ::-1].copy()
            if np.ptp(rgb, axis=(0, 1)).max() == 0:
                raise RuntimeError('RGB camera returned a spatially uniform/empty image')
            if frame_ids and frame.frame <= frame_ids[-1]:
                raise RuntimeError('RGB camera repeated or reordered a frame')
            frame_ids.append(frame.frame)
            hashes.add(hashlib.sha256(rgb.tobytes()).hexdigest())
            Image.fromarray(rgb).save(folder / f'{index:05d}.png')
        finish = vehicle.get_location()
        distance = float(np.linalg.norm([finish.x-start.x, finish.y-start.y, finish.z-start.z]))
        if len(hashes) < 15 or distance < 0.5:
            raise RuntimeError(f'No moving camera sequence: {len(hashes)} unique frames, {distance:.2f}m movement')
        result = {'status': 'PASSED', 'frames': len(frame_ids), 'frame_ids': frame_ids,
                  'unique_frames': len(hashes), 'vehicle_distance_m': distance,
                  'video': {'status': 'SKIPPED', 'reason': 'deactivated'}}
        if render_video:
            video = output / 'carla-camera.mp4'
            encoding = diagnostic(['ffmpeg', '-y', '-framerate', '10', '-i', str(folder / '%05d.png'),
                '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-movflags', '+faststart', str(video)], output / 'video.log')
            if encoding is None or encoding.returncode:
                raise RuntimeError('Video encoding failed; see video.log')
            result['video'] = {'status': 'PASSED', 'file': video.name, 'frames': 30, 'fps': 10, 'seconds': 3}
        return result
    finally:
        if camera is not None:
            camera.stop()
            camera.destroy()
        vehicle.destroy()


root, report, mode = Path(sys.argv[1]), Path(sys.argv[2]), sys.argv[3]
settings = json.loads((root / 'carla-only-settings.json').read_text())
output = report / mode
output.mkdir()
info = {'status': 'FAILED', 'mode': mode, 'phase': 'install', 'rpc': 'SKIPPED',
        'rgb': {'status': 'SKIPPED'}, 'map': settings['map']}
process = None
engine_log = None
env = os.environ.copy()
started = time.monotonic()
try:
    try:
        user = pwd.getpwnam('carla-colab')
    except KeyError:
        subprocess.run(['useradd', '--create-home', '--shell', '/bin/bash', 'carla-colab'], check=True)
        user = pwd.getpwnam('carla-colab')
    subprocess.run(['usermod', '-a', '-G', 'video', user.pw_name], check=True)
    runtime = Path(user.pw_dir) / 'runtime'
    runtime.mkdir(mode=0o700, exist_ok=True)
    os.chown(runtime, user.pw_uid, user.pw_gid)
    for key in ('HF_TOKEN', 'SDL_VIDEODRIVER', 'DISPLAY', 'LD_PRELOAD'):
        env.pop(key, None)
    env.update(HOME=user.pw_dir, XDG_RUNTIME_DIR=str(runtime))
    as_user = ['runuser', '-u', user.pw_name, '--']
    server = root / 'carla-0.9.16'
    if not (server / '.installed').exists():
        if shutil.disk_usage(root).free < 35e9:
            raise RuntimeError('Need at least 35 GB free disk for CARLA')
        archive = root / 'CARLA_0.9.16.tar.gz'
        subprocess.run(['curl', '--fail', '--location', '--retry', '3', '--output', str(archive),
                        'https://downloads.carlasim.com/Linux/CARLA_0.9.16.tar.gz'], check=True)
        server.mkdir(exist_ok=True)
        subprocess.run(['tar', '-xzf', str(archive), '-C', str(server), '--no-same-owner'], check=True)
        subprocess.run(['chown', '-R', f'{user.pw_uid}:{user.pw_gid}', str(server)], check=True)
        (server / '.installed').touch()
        archive.unlink()
    info['phase'] = 'dependencies'
    binary = server / 'CarlaUE4/Binaries/Linux/CarlaUE4-Linux-Shipping'
    subprocess.run([*as_user, 'chmod', '+x', str(binary)], check=True)
    dependencies = diagnostic([*as_user, 'ldd', str(binary)], output / 'dependencies.log', env)
    if dependencies is None or dependencies.returncode or 'not found' in dependencies.stdout:
        raise RuntimeError('Missing engine dependencies; see dependencies.log')
    rendering = ['-nullrhi', '-RenderOffScreen']
    if mode == 'gpu-rgb':
        info['phase'] = 'vulkan'
        driver_probe = diagnostic(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], output / 'driver.log')
        if driver_probe is None or driver_probe.returncode:
            raise RuntimeError('NVIDIA GPU unavailable; choose a GPU runtime, see driver.log')
        driver = driver_probe.stdout.splitlines()[0].strip()
        if not all(part.isdigit() for part in driver.split('.')):
            raise ValueError(f'Unexpected driver version: {driver}')
        library = ctypes.util.find_library('EGL_nvidia')
        api_version = '1.2.0'
        if not library:
            graphics = root / f'nvidia-graphics-{driver}'
            library = str(graphics / f'libEGL_nvidia.so.{driver}')
            if not Path(library).exists():
                runfile = root / f'NVIDIA-{driver}.run'
                subprocess.run(['curl', '--fail', '--location', '--retry', '3', '--output', str(runfile),
                    f'https://us.download.nvidia.com/tesla/{driver}/NVIDIA-Linux-x86_64-{driver}.run'], check=True)
                subprocess.run(['/bin/sh', str(runfile), '--extract-only', '--target', str(graphics)], check=True)
                subprocess.run(['chown', '-R', f'{user.pw_uid}:{user.pw_gid}', str(graphics)], check=True)
                runfile.unlink()
            as_user += ['env', f"LD_LIBRARY_PATH={graphics}:{env.get('LD_LIBRARY_PATH', '')}"]
            descriptor = graphics / 'nvidia_icd.json'
            if descriptor.exists():
                api_version = json.loads(descriptor.read_text())['ICD']['api_version']
        icd = Path(user.pw_dir) / 'carla-only-icd.json'
        icd.write_text(json.dumps({'file_format_version': '1.0.0', 'ICD': {'library_path': library, 'api_version': api_version}}))
        env['VK_ICD_FILENAMES'] = str(icd)
        env['VK_LOADER_DEBUG'] = 'error,warn'
        info.update(vulkan_icd=library, vulkan_api_version=api_version)
        vulkan = diagnostic([*as_user, 'vulkaninfo', '--summary'], output / 'vulkan.log', env)
        if vulkan is None or vulkan.returncode or not any('deviceName' in line and 'NVIDIA' in line for line in vulkan.stdout.splitlines()):
            raise RuntimeError('No NVIDIA Vulkan device; see vulkan.log')
        rendering = ['-RenderOffScreen', '-vulkan']
    info['phase'] = 'launch'
    try:
        with socket.create_connection(('127.0.0.1', 2000), timeout=1):
            raise RuntimeError('Port 2000 already in use; stop the existing server or reset the Colab runtime')
    except OSError:
        pass
    engine_log = Path(user.pw_dir) / f'{report.name}-{mode}-engine.log'
    command = [*as_user, str(binary), 'CarlaUE4', f"/Game/Carla/Maps/{settings['map']}",
        *rendering, '-nosound', '-unattended', '-stdout', '-FullStdOutLogOutput', '-FORCELOGFLUSH',
        f'-abslog={engine_log}', '-carla-rpc-port=2000']
    info['command'] = command
    print('Starting', mode, settings['map'], flush=True)
    with (output / 'server.log').open('w') as log:
        process = subprocess.Popen(command, cwd=server, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        client = carla.Client('127.0.0.1', 2000)
        client.set_timeout(2)
        world = wait_for_server(process, client, settings['startup_timeout'], info)
        info['world'] = world.get_map().name
        if mode == 'gpu-rgb':
            client.set_timeout(30)
            info['phase'] = 'rgb'
            info['rgb'] = {'status': 'FAILED'}
            info['rgb'] = test_rgb(world, output, settings['render_video'])
        info.update(status='PASSED', phase='complete')
except Exception as error:
    info.update(error=str(error), traceback=traceback.format_exc())
    print(info['traceback'], flush=True)
finally:
    if process is not None:
        diagnostic(['nvidia-smi'], output / 'gpu-at-end.log')
        diagnostic(['ps', '-eo', 'pid,ppid,stat,pcpu,pmem,wchan:32,comm'], output / 'processes-at-end.log')
        try:
            stop_server(process)
        except OSError as error:
            info.update(status='FAILED', cleanup_error=str(error))
    if engine_log is not None and engine_log.exists():
        shutil.copyfile(engine_log, output / 'engine.log')
    info['seconds'] = round(time.monotonic() - started, 2)
    (output / 'result.json').write_text(json.dumps(info, indent=2))
print(json.dumps(info, indent=2))
sys.exit(0 if info['status'] == 'PASSED' else 1)
'''
(ROOT / 'carla-only-worker.py').write_text(CARLA_TEST)

## 3. CPU-Baseline: Engine, RPC und Weltzugriff

Hier ist der Renderer aus. PASSED cpu-rpc ist ausschließlich ein Engine-/API-Test.
Der eigene Server wird danach beendet, bevor der GPU-Test beginnt.

In [ ]:
if SETUP_OK:
    run_mode('cpu-rpc')
else:
    RESULTS['cpu-rpc'] = {'status': 'SKIPPED', 'reason': 'Setup failed'}

## 4. GPU-Rendering, Kameraframes und kurze Fahrt

Der Test läuft auch dann, wenn die CPU-Baseline fehlgeschlagen ist. Fehlende oder
einfarbige Kamerabilder gelten als Fehler. Ein Erfolg erzeugt bei aktiviertem Video
gpu-rgb/carla-camera.mp4: drei Sekunden echter CARLA-Kamerabilder, ohne Modellpfad.

In [ ]:
if SETUP_OK:
    run_mode('gpu-rgb')
    if RESULTS['gpu-rgb']['status'] == 'PASSED' and RENDER_VIDEO:
        from IPython.display import Video, display
        display(Video(str(REPORT / 'gpu-rgb/carla-camera.mp4'), embed=True))
else:
    RESULTS['gpu-rgb'] = {'status': 'SKIPPED', 'reason': 'Setup failed'}

## 5. Bericht herunterladen

Diese Zelle lässt sich auch nach einem abgebrochenen Test einzeln ausführen.
summary.json enthält die Ergebnisse. Pro Test liegen result.json, Server-/Engine-
Logs sowie die GPU-Belegung und Prozesszustände vor dem Abschalten vor, soweit der
jeweilige Schritt erreicht wurde. Installation und Client-Umgebung bleiben außerhalb des ZIPs.

Wenn **cpu-rpc erfolgreich**, aber **gpu-rgb fehlgeschlagen** ist, konzentriert sich die
weitere Fehlersuche auf Grafikinitialisierung oder Sensoren. Wenn auch cpu-rpc
scheitert, ist bereits Engine/API-Start betroffen. Ein bestandener CPU-Test ersetzt
keinen bestandenen GPU-Test.

In [ ]:
for name in ('cpu-rpc', 'gpu-rgb'):
    RESULTS.setdefault(name, {'status': 'SKIPPED', 'reason': 'Test cell not reached'})
    result = RESULTS[name]
    print(result['status'], name, '| phase:', result.get('phase'), '| error:', result.get('error', ''))
summary = {'carla_version': '0.9.16', 'map': MAP, 'checks': RESULTS}
(REPORT / 'summary.json').write_text(json.dumps(summary, indent=2))
archive = shutil.make_archive(str(ROOT / 'carla-only-report'), 'zip', REPORT)
from google.colab import files
files.download(archive)